<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_7_Memory_Management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 7 — Why AI Forgets Everything (and How to Fix It)


### 🎬 Today's real-time scenario

Meet **Priya**. She runs **TastyTiffin**, a tiffin (home-style lunch) delivery startup in Chennai. She just launched a support chatbot called **Mitra** ("friend") built on the Claude API.

On day one, a customer types:

> Customer: *"Hi, I'm Karthik. My order #4521 arrived cold."*
> Mitra: *"So sorry, Karthik! I've noted order #4521."*
> Customer: *"So what will you do about my order?"*
> Mitra: *"Which order? Could you share your order number?"* 😱

The customer is furious. Mitra forgot **everything** in 10 seconds. Priya thinks the AI is broken.

**It is not broken. It is stateless — and today you will learn exactly what that means, why it happens, and three professional ways to fix it.** By the end, you will rebuild Mitra so it remembers, handles long conversations, and costs up to 90% less to run.


### What you'll be able to do after this session

- **Explain** why every LLM API call starts with zero memory (stateless vs stateful)
- **Implement** conversation memory by hand with a Python list
- **Build** a chatbot class that remembers, using the real Claude API
- **Manage** long conversations with a sliding window (without breaking the API rules)
- **Cut costs** with prompt caching — and read the cache numbers in the API response
- **Architect** a production support bot and defend your choices in an interview

# Foundations — 4 words you need first

**Token** — the small chunks of text a model reads and writes. Roughly, 1 token ≈ 3.5 English characters, or about 3/4 of a word. "TastyTiffin delivers hot lunches" is about 7 tokens.

**Context window** — the maximum number of tokens the model can read *in one request* (your input + its output). Think of it as the model's desk size: anything on the desk it can see perfectly; anything not on the desk does not exist for it.

**Stateless** — the system keeps *no memory* between requests. Every request starts from zero.

**Stateful** — the system *does* carry information from one request to the next. (State = stored information.)

### Today's map

```
The problem            The fixes
-----------            ---------------------------------
Claude forgets   -->   1. Send the history back (memory list)
                       2. Trim old messages (sliding window)
                       3. Stop re-paying for the same text (prompt caching)
```

One problem, three tools. Let's set up and see the problem live.


In [ ]:
# Setup cell 1 — install the Anthropic SDK
!pip install anthropic

In [ ]:
# Setup cell 2 — key + client (Colab Secret named MY_API_KEY)
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # fast + cheap — perfect for learning
print("Ready ✅")

# Section 1 — The Memory Problem: Why Claude Forgets

### 🧠 What is it?

Every call to the Claude API is a **brand-new meeting**. The API does not save your previous messages. Send a request, get a reply, and the moment the reply is sent — from the API's point of view, the conversation never happened.

### Why does it matter?

Because *every* chatbot, support agent, and AI copilot needs memory to be useful. Mitra forgetting Karthik's order number is not a bug in Claude — it is a missing piece in **Priya's app**. Whoever builds the app owns the memory. That is a core job of an AI engineer.

### How does it work?

1. Your app sends a request: a list of messages.
2. Claude reads **only what is in that request** and writes a reply.
3. The request is processed and the reply returned. No conversation state is kept on the server for your next call.
4. Next request? Claude sees **only** what that new request contains.

### An analogy that sticks

Claude is like a brilliant consultant with **no long-term memory between meetings**. Inside one meeting (one request), the consultant is sharp and remembers everything said. But walk out and walk back in, and you must re-brief them from page one. The briefing document you hand them each time = the `messages` list.

### When is stateless actually good?

- **Privacy** — your conversation is not stored on the model side waiting to leak into someone else's chat.
- **Scale** — any server can handle any request, since no request depends on server-side conversation state. (This is the same reason the web's HTTP protocol is stateless.)
- **Predictability** — the model's answer depends only on what you sent, so bugs are reproducible.

> **Accuracy note:** "Claude forgets" means the **API keeps no conversation state between calls**. It does *not* mean your data vanishes from the universe instantly — API logs and safety systems are separate topics. And on claude.ai the *app* replays your history for you, which is why the website "remembers" but a raw API call does not. Interviewers love this distinction.

### 🎤 Interview angle

**Q: "Is Claude stateless or stateful?"**
*Model answer:* "The Messages API is stateless — each request is independent and contains the full conversation. Statefulness is built in the application layer: my app stores the history and resends it every turn. Products like claude.ai feel stateful because the app does exactly that."

### ❌ Don't mix these up

- ❌ "Claude remembers my last API call." → ✅ It sees only what's inside the current request.
- ❌ "Stateless means the model is dumb." → ✅ Stateless is a deliberate design for privacy and scale; the intelligence is unchanged.
- ❌ "claude.ai remembers, so the API must too." → ✅ The claude.ai *app* resends your history each turn — same trick you'll build today.

### 🤯 Fun fact

The web itself has the same "amnesia": HTTP is a stateless protocol. In 1994, Netscape engineer **Lou Montulli** invented the browser **cookie** so websites could remember you between page loads. Today you're going to invent the "cookie" for your chatbot.


## 💻 Lab 1 — Prove that Claude forgets

**Objective:** make two separate API calls and watch the second one fail to remember the first.


In [ ]:
# Lab 1: two SEPARATE calls — no shared history

# --- Call 1: tell Claude a fact ---
reply_1 = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[{"role": "user", "content": "Hi! I'm Karthik and my order number is 4521."}],
)
print("Call 1:", reply_1.content[0].text)

print("-" * 60)

# --- Call 2: a brand-new request. Notice: we send ONLY the new question. ---
reply_2 = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[{"role": "user", "content": "What is my order number?"}],
)
print("Call 2:", reply_2.content[0].text)

**Expected result:** in Call 2, Claude says it doesn't know your order number (or asks you for it). It is not being difficult — the request literally contained no mention of Karthik or #4521.

**Try this 🔧:** change Call 1 to tell Claude your favourite food, then ask about it in Call 2. Same amnesia, every time.

### ✅ Checkpoint

> **Q:** Whose job is it to make Mitra remember Karthik — Anthropic's or Priya's?

<details><summary>Show answer</summary>

**Priya's (the app builder's).** The API is stateless by design. The application stores the history and sends it back with every request.
</details>


# Section 2 — The Fix: Send the History Back (Stateful Chat)

### 🧠 What is it?

The fix is almost embarrassingly simple: **keep a list of every message — yours and Claude's — and send the whole list every time.** Claude reads the full conversation fresh on each call and replies as if it remembered all along.

### Why does it matter?

This one pattern powers claude.ai, ChatGPT, and virtually every AI chat product you have ever used. Master this list and you understand the memory layer of a billion-dollar product category.

### How does it work?

Each message is a small dictionary with two keys:

```python
{"role": "user", "content": "My order #4521 arrived cold."}
{"role": "assistant", "content": "So sorry! I've noted order #4521."}
```

- `role` — who is speaking: `"user"` (the human) or `"assistant"` (Claude).
- `content` — what they said.

The rules the API enforces: the conversation should **start with a `user` message**, and roles should **alternate** user → assistant → user → assistant. Remember this rule — it will matter again in the sliding-window section.

### Stateless vs stateful, side by side

```
STATELESS (broken Mitra)                 STATEFUL (fixed Mitra)
------------------------                 ----------------------
Request 1: [msg1]                        Request 1: [msg1]
Request 2: [msg2]     <- knows nothing   Request 2: [msg1, reply1, msg2]
Request 3: [msg3]     <- knows nothing   Request 3: [msg1, reply1, msg2, reply2, msg3]
```

Notice the stateful side: every request re-sends **everything so far**. That has a cost — we deal with it in Sections 4 and 5.

### 💡 Remember this

> *"LLM memory is an illusion your app creates: the model never remembers — your code re-tells it the whole story on every single call."*

### 🎤 Interview angle

**Q: "How would you add memory to an LLM chatbot?"**
*Model answer:* "Store every user and assistant message in order, and include the full list in the `messages` array of each new request. The model re-reads the conversation each turn. Then manage growth with truncation/sliding windows, summarization, and prompt caching to control cost."


## 💻 Lab 2 — The conversation list, by hand

**Objective:** build the history list manually and prove Claude now "remembers".


In [ ]:
# Lab 2: one growing list = memory

history = []   # the entire memory of our chatbot lives in this list

# --- Turn 1 ---
history.append({"role": "user", "content": "Hi! I'm Karthik and my order number is 4521."})

reply = client.messages.create(model=MODEL, max_tokens=100, messages=history)
answer_1 = reply.content[0].text
print("Turn 1:", answer_1)

# Save Claude's reply into the same list — this is the crucial step!
history.append({"role": "assistant", "content": answer_1})

print("-" * 60)

# --- Turn 2: append the new question and send the WHOLE list ---
history.append({"role": "user", "content": "What is my order number?"})

reply = client.messages.create(model=MODEL, max_tokens=100, messages=history)
print("Turn 2:", reply.content[0].text)   # Claude now knows: 4521 ✅

**Expected result:** Turn 2 correctly answers "4521". Same model, same API — the only difference is *what we sent*.

**Try this 🔧:** print `history` after Turn 2 and count the messages. Then comment out the `history.append({"role": "assistant"...})` line and rerun — watch quality drop, because Claude no longer sees its own earlier reply.

### ✅ Checkpoint

> **Q:** Why must we also save Claude's *replies* in the list, not just our questions?

<details><summary>Show answer</summary>

Because Claude needs to see **both sides** of the conversation to stay consistent. If it promised Karthik a refund in Turn 2, it must see that promise in Turn 3 — otherwise it may contradict itself.
</details>


# Section 3 — Packaging It: Mitra v1, a Chatbot Class

Doing `append` by hand every turn is error-prone (forget one line and memory silently breaks). Professionals wrap the pattern in a small class. Same idea, safer to use — and this is the shape you'll see in real codebases.


In [ ]:
# Mitra v1: a chatbot with memory, in ~20 lines

class MitraBot:
    def __init__(self, system_prompt):
        self.system_prompt = system_prompt   # standing instructions (not part of messages)
        self.history = []                    # the memory list

    def chat(self, user_text):
        self.history.append({"role": "user", "content": user_text})
        reply = client.messages.create(
            model=MODEL,
            max_tokens=200,
            system=self.system_prompt,   # system prompt rides along on every call
            messages=self.history,
        )
        answer = reply.content[0].text
        self.history.append({"role": "assistant", "content": answer})
        return answer

mitra = MitraBot("You are Mitra, the friendly support assistant for TastyTiffin, "
                 "a tiffin delivery service in Chennai. Be warm and brief.")

print("1:", mitra.chat("Hi, I'm Karthik. My order #4521 arrived cold."))
print("2:", mitra.chat("What will you do about it?"))
print("3:", mitra.chat("Remind me — what was my order number and my complaint?"))

**Expected result:** turn 3 recalls both the order number **and** the complaint. Priya's angry customer is now a happy customer.

### ✅ Checkpoint

> **Q:** Where exactly is the conversation stored — in Claude, or somewhere else?

<details><summary>Show answer</summary>

In **your code** — the `mitra.history` list, living in your program's memory. Claude stores nothing between calls. If your program restarts, the memory is gone (real apps save it in a database).
</details>

### 🏢 Enterprise perspective

This is exactly how enterprise assistants persist chats: the history list is saved per-user in a database (Postgres, DynamoDB, Redis), reloaded when the user returns, and replayed to the model. "Conversation history" in your banking app's chatbot = rows in a database being turned back into a `messages` list.


# Section 4 — The Catch: Conversations Grow (Tokens & the Context Window)

### 🧠 What is it?

Our fix has a hidden bill. Every turn re-sends the **whole** history, so requests get bigger and bigger. Two limits push back:

1. **The context window** — the model's hard reading limit per request. For our lab model, **Claude Haiku 4.5, that's 200,000 tokens** (about 150,000 words, roughly a 400-page book).
2. **Cost** — the API charges **per input token, every call**. Turn 50 re-sends turns 1–49. You pay for turn 1's text *fifty times* over the conversation.

> **Accuracy note:** 200K is the figure for Haiku 4.5, our lab model. Newer models go further — Claude Sonnet 4.6/5 and Opus 4.8 support a **1-million-token** context window. Bigger windows raise the ceiling, but the "you re-pay for history every call" cost problem remains — which is why the next two sections exist.

### Why does it matter (with real money)

Claude Haiku 4.5 pricing: **USD 1 per million input tokens, USD 5 per million output tokens**. Sounds tiny — until Mitra serves 10,000 customers a day and each conversation re-sends a growing history plus a long system prompt on every turn. Memory management is a **cost-engineering skill**, not just a correctness skill.

### How do I measure tokens? (the professional way)

Guessing is fine for intuition (≈3.5 characters per token in English), but Anthropic gives you a real **token counting endpoint** — `client.messages.count_tokens(...)` — which returns the exact count *without* running the model. It's free to use (rate limits apply).

### 💡 Remember this

> *"The context window limits how much the model can read at once; your wallet limits how often you should make it re-read everything."*


## 💻 Lab 3 — Count tokens for real, and watch a conversation swell

**Objective:** use the real token-counting API to see exactly how history growth costs you.


In [ ]:
# Lab 3a: exact token counting (no model run, no output cost)

count = client.messages.count_tokens(
    model=MODEL,
    messages=[{"role": "user", "content": "My order #4521 arrived cold."}],
)
print("Exact input tokens:", count.input_tokens)

In [ ]:
# Lab 3b: watch the per-turn input size GROW as history accumulates

demo_history = []
questions = [
    "Hi, I'm Karthik. My order #4521 arrived cold and I want a refund.",
    "Also, can you check why delivery took 90 minutes?",
    "And update my address to 12, Gandhi Street, Adyar, Chennai.",
    "Actually, make the refund a wallet credit instead.",
]

for i, q in enumerate(questions, start=1):
    demo_history.append({"role": "user", "content": q})
    # pretend-assistant reply so the history looks like a real chat
    demo_history.append({"role": "assistant", "content": f"Noted! (reply to turn {i})"})
    count = client.messages.count_tokens(model=MODEL, messages=demo_history)
    print(f"After turn {i}: sending {count.input_tokens:>4} input tokens per call")

**Expected result:** the token count climbs every turn — and you'd pay for the *entire* number on *every* call. Multiply by thousands of users and the problem is obvious.

**Try this 🔧:** append one giant message (paste a long paragraph 20 times) and re-count. Now imagine a customer pasting their whole order history into chat.

### ✅ Checkpoint

> **Q (True/False):** With conversation-list memory, the cost of turn N includes the tokens of all previous turns.

<details><summary>Show answer</summary>

**True.** Every request re-sends the full history, and input tokens are billed on every request. This is the single most important cost fact in this session.
</details>


# Section 5 — Fix #1: The Sliding Window

### 🧠 What is it?

A **sliding window** keeps only the **most recent N messages** and drops the oldest ones. Like a car's side mirror: you see what's close behind you clearly; the road far behind is gone.

### Why does it matter?

It caps both problems at once: the history can never exceed the window (so you never hit the context limit), and per-turn cost stops growing.

### How does it work?

```
Window = keep last 6 messages

Before: [m1, m2, m3, m4, m5, m6, m7, m8]     (8 messages - too many)
After:  [        m3, m4, m5, m6, m7, m8]     (oldest 2 dropped)
```

### When should I avoid it (the tradeoff)?

The window **forgets facts**, not just text. If Karthik gave his order number in message 1 and you drop message 1, Mitra forgets the order number — the original bug returns through the back door. Production systems therefore combine a window with either (a) a running **summary** of dropped messages, or (b) extracting key facts (name, order #) into the **system prompt**, which never gets dropped.

> ⚠️ **The classic beginner bug:** the API expects the conversation to **begin with a `user` message** with roles alternating. If you trim an odd number of messages, your history may now *start with an assistant message* → API error. Rule: **trim in user+assistant pairs** (keep an even count, oldest pair out first).

### 💡 Remember this

> *"A sliding window trades old context for a flat cost — never trim it in a way that breaks the user-first, alternating-roles rule."*

### ❌ Don't mix these up

- ❌ "The sliding window compresses old messages." → ✅ It **deletes** them. Compression (summarization) is a different, complementary technique.
- ❌ "Trim by any count that fits." → ✅ Trim in pairs so the list still starts with a `user` turn.
- ❌ "Bigger window = always better." → ✅ Bigger window = higher cost per turn and more for the model to wade through. Choose the smallest window that keeps answer quality.


## 💻 Lab 4 — A sliding window that doesn't break the rules

**Objective:** add a pair-safe sliding window to Mitra and watch old turns fall away.


In [ ]:
# Lab 4: pair-safe sliding window

def sliding_window(history, max_messages=6):
    """Keep only the most recent messages, trimming in PAIRS
    so the list always starts with a 'user' message."""
    if len(history) <= max_messages:
        return history
    excess = len(history) - max_messages
    if excess % 2 == 1:          # never trim an odd number
        excess += 1              # round up to a full user+assistant pair
    print(f"  [window] dropping {excess} oldest messages")
    return history[excess:]

class MitraBotV2(MitraBot):                      # reuse everything from v1
    def chat(self, user_text):
        self.history = sliding_window(self.history, max_messages=6)
        return super().chat(user_text)

mitra2 = MitraBotV2("You are Mitra, TastyTiffin's support assistant. Be warm and brief.")

print("1:", mitra2.chat("Hi, I'm Karthik, order #4521 arrived cold."), "\n")
print("2:", mitra2.chat("I'd like a refund please."), "\n")
print("3:", mitra2.chat("Also what time do you deliver dinner?"), "\n")
print("4:", mitra2.chat("And do you have a veg-only menu?"), "\n")
print("5:", mitra2.chat("What was my order number?"))   # turn 1 may be gone by now!

**Expected result:** by turn 5 the window has dropped the earliest pair(s). Mitra may no longer know "#4521" — **exactly** the tradeoff we predicted. Seeing a technique's failure mode is as important as seeing it work.

**Try this 🔧:** raise `max_messages` to 10 and rerun — turn 5 now remembers. Then add the order number to the system prompt instead, keep the window at 6, and watch it survive trimming. That's the "important facts live in the system prompt" pattern.

### 🎤 Interview angle

**Q (architecture): "Your chatbot hits the context limit in long support chats. Options?"**
*Model answer:* "Three levers, often combined: a pair-safe sliding window for recency; summarization — replace dropped turns with a short model-written summary; and fact extraction — pin durable facts (name, order ID, preferences) into the system prompt or a profile store. Choice depends on how much old detail the use case truly needs."


# Section 6 — Fix #2: Prompt Caching (Stop Re-Paying for the Same Text)

### 🧠 What is it?

Look at what Mitra re-sends **identically** on every single call: the system prompt, the menu, the refund policy, and all the old conversation turns. **Prompt caching** tells Anthropic's servers: *"I'll be sending this exact beginning again — keep your processed version warm."* On the next call, the cached part is reused instead of re-processed.

To be precise about what's stored: the API keeps the model's internal *processed state* for that prefix (and a hash to match it) in memory for a short time — not a permanent copy of your chat.

### Why does it matter? (the numbers, verified)

| What | Price on Haiku 4.5 |
|---|---|
| Normal input tokens | USD 1.00 / million |
| Writing to cache (first call) | USD 1.25 / million (1.25× — small surcharge) |
| **Reading from cache (later calls)** | **USD 0.10 / million (0.1× — 90% off!)** |

Anthropic's own launch numbers: up to **90% cost reduction** and up to **85% latency reduction** on long prompts. For Priya, the 2,000-token menu+policy block that every customer request re-sends now costs one-tenth as much from the second call onward — *and* responses start faster.

### How does it work?

1. You mark a **cache breakpoint** in your request with `cache_control: {"type": "ephemeral"}`.
2. First call: everything up to the breakpoint is processed and **written** to the cache (1.25× price).
3. Later calls with the **exact same prefix**: it's **read** from cache (0.1× price). Each read also refreshes the timer, free.
4. The cache lives **5 minutes** by default (a 1-hour option costs 2× to write). After that it expires — remember, this is a *cost* optimization, not a memory system!

### The four rules that make or break caching

1. **Exact match only.** One changed character in the cached part = cache miss. Put *stable* content (system prompt, tools, policies) first, *changing* content (the new question) last.
2. **Order is fixed:** tools → system → messages. A change higher up invalidates everything below it.
3. **Minimum size.** Below a per-model minimum, nothing is cached at all (no error — it just silently doesn't cache). **On Haiku 4.5 the minimum is 4,096 tokens.** Our lab builds a big enough prompt on purpose.
4. **Put the breakpoint on content that doesn't change.** Mark the end of the *stable* prefix — not the user's new message, which changes every call and can never match.

### How do I know it worked? Read the receipt

Every response includes `usage` fields — your cache receipt:

- `cache_creation_input_tokens` — tokens **written** to cache this call
- `cache_read_input_tokens` — tokens **read** from cache this call (the 90%-off part)
- `input_tokens` — tokens after the breakpoint, billed normally

### 💡 Remember this

> *"Caching doesn't make Claude remember — it makes re-sending what you were already re-sending 90% cheaper and much faster. Memory is your job; caching is your discount."*

### ❌ Don't mix these up

- ❌ "Prompt caching gives Claude long-term memory." → ✅ It's a cost/latency optimization on content **you still send every time**. Memory stays your app's job.
- ❌ "Cached responses are lower quality." → ✅ Output is **identical** with or without caching — only price and speed change.
- ❌ "I cached my 500-token system prompt on Haiku." → ✅ Below the 4,096-token minimum on Haiku 4.5, nothing is cached (silently). Check the usage fields.

### 🤯 Fun fact

Anthropic launched prompt caching on **August 14, 2024**. One highlighted use case: caching an *entire book*. Anthropic's example showed that chatting with the full text of *Pride and Prejudice* cached went from ~12 seconds of latency to ~2.4 seconds — an 80% speedup, plus 90% off the input bill.


## 💻 Lab 5 — See the 90% discount in the usage receipt

**Objective:** build a TastyTiffin policy pack big enough to cache (>4,096 tokens on Haiku 4.5), mark it with a cache breakpoint, and watch the usage numbers flip from *write* to *read*.


In [ ]:
# Lab 5a: build a policy pack big enough to cross Haiku 4.5's 4,096-token cache minimum

policy_chunk = """\
TASTYTIFFIN OPERATIONS, REFUND AND SERVICE POLICY HANDBOOK
Document ID: TT-POL-2026-11
Effective date: 01 April 2026
Supersedes: TT-POL-2025-07, TT-POL-2025-09, TT-POL-2026-03
Owner: Customer Experience and Fulfilment Council, TastyTiffin Foods Private Limited, Chennai
Review cadence: quarterly, with emergency amendment allowed for food safety incidents
Applicable geography: Chennai Metropolitan Area and notified peripheral pincodes
Applicable channels: TastyTiffin mobile application (Android and iOS), TastyTiffin web ordering portal, TastyTiffin WhatsApp ordering bot, corporate bulk ordering desk, and telephone ordering through the central helpline

SECTION 0. PURPOSE AND SCOPE

0.1 This handbook is the single authoritative source of truth for refund adjudication, delivery service levels, menu composition, food quality standards, compensation ladders, subscription handling, and escalation procedure for TastyTiffin. Where any prior circular, WhatsApp broadcast, rider briefing, kitchen notice board instruction, or verbal assurance conflicts with this handbook, this handbook prevails.

0.2 This handbook is written to be readable by three audiences simultaneously. First, customer support associates who need a deterministic answer within ninety seconds of a customer contact. Second, kitchen and dispatch supervisors who need to understand which operational failures translate into financial liability. Third, automated systems, including retrieval augmented assistants, which consume this document as grounding context and must not invent policy that is not written here.

0.3 If a situation is not covered by this handbook, the associate must not improvise a monetary outcome. The associate must apply Section 14, the unspecified-scenario procedure, which routes the case to a supervisor with a provisional goodwill cap.

0.4 Nothing in this handbook creates an obligation on TastyTiffin that exceeds the total value of the affected order plus the compensation ceilings expressly stated herein, except where statutory consumer protection law requires otherwise.

SECTION 1. DEFINITIONS

1.1 "Order" means a single confirmed transaction with a unique order identifier, containing one or more line items, placed through any of the applicable channels listed above.

1.2 "Line item" means one saleable unit within an order. A single meal box, one add-on, one beverage, or one dessert is each a distinct line item. A combo is a single line item unless the combo is explicitly decomposed at checkout.

1.3 "Delivery timestamp" means the moment the rider marks the order as delivered in the rider application, or the moment the delivery one-time password is validated, whichever occurs first. Where both are absent, the delivery timestamp is deemed to be the geofence exit timestamp of the rider device from the delivery address polygon.

1.4 "Promised window" means the delivery window shown to the customer at the moment of checkout confirmation, not the window shown while browsing. Browsing estimates are indicative and carry no service level obligation.

1.5 "Late delivery" means a delivery timestamp that occurs more than forty-five minutes after the closing edge of the promised window, subject to the exclusions in Section 6.

1.6 "Cold food" means a meal delivered below the minimum service temperature defined for that dish in Section 4, verified by any one of the three evidence methods described in Section 3.

1.7 "Wallet credit" means non-withdrawable store value issued to the customer's TastyTiffin account, usable against future orders, with the validity and stacking rules described in Section 8.

1.8 "Refund to source" means reversal of the paid amount to the original payment instrument, including unified payments interface handles, credit cards, debit cards, net banking, and corporate invoice adjustments.

1.9 "Coupon" means a percentage or absolute discount code applied at checkout, with the constraints described in Section 8.

1.10 "Meal integrity failure" means any defect that renders the meal unfit or materially unpleasant to consume, including but not limited to spillage, contamination, foreign object presence, staleness, curdling, gravy separation beyond acceptable tolerance, or crushed packaging.

1.11 "Serviceable pincode" means a Chennai pincode listed in Section 5 as active for the relevant meal period. A pincode may be active for lunch and inactive for dinner, or the reverse.

1.12 "Subscription" means a recurring meal plan with weekly or monthly billing, described in Section 9.

1.13 "Goodwill" means a discretionary issuance not mandated by any clause of this handbook, capped and logged as described in Section 14.

1.14 "Repeat claimant" means an account that has raised three or more compensable claims within any rolling thirty day period. Repeat claimant handling is described in Section 12.

1.15 "Meal period" means either the lunch service or the dinner service as defined in Section 5.

SECTION 2. THE HEADLINE COMMITMENTS

2.1 Cold food commitment. A meal delivered cold qualifies for a full refund of that line item, or wallet credit of equal value at the customer's election, when reported within twenty four hours of the delivery timestamp. The customer chooses the settlement form. The associate must not steer the customer toward wallet credit, and must not present wallet credit as the only option. If the customer expresses no preference, the default is refund to source for prepaid orders and wallet credit for cash on delivery orders, because cash reversal requires a manual treasury run.

2.2 Late delivery commitment. A delivery that exceeds forty five minutes beyond the promised window earns a twenty percent coupon for the customer's next order. The coupon is issued automatically by the fulfilment engine without the customer needing to ask. If automatic issuance fails, any associate may issue it manually on request with no further approval.

2.3 The two commitments in clauses 2.1 and 2.2 are independent and cumulative. A meal that arrives both late and cold attracts both the refund or credit for the cold food and the twenty percent coupon for lateness. Associates must not treat one as absorbing the other. This is the single most frequently mishandled scenario in audit sampling and is called out here deliberately.

2.4 Neither commitment requires the customer to return the food, photograph the food, or dispose of the food under supervision. Evidence requirements are graduated by claim value as described in Section 3, and low value claims are settled on the customer's word alone.

SECTION 3. EVIDENCE STANDARDS AND ADJUDICATION TIERS

3.1 TastyTiffin operates a three tier evidence model. The tier is determined by the rupee value of the claim and the claim history of the account, never by the tone of the customer.

3.2 Tier one, trust settled. Applies where the claim value is at or below three hundred rupees and the account is not a repeat claimant. No evidence is required. The associate settles immediately on the customer's statement. Target handling time is under ninety seconds. Approximately seventy one percent of all TastyTiffin claims settle in tier one.

3.3 Tier two, light evidence. Applies where the claim value is between three hundred and one and eight hundred rupees, or where the account has raised two claims in the trailing thirty days. Any one of the following satisfies tier two: a photograph of the meal as received, a photograph of the packaging seal, a description that is consistent with a known dispatch anomaly logged for that batch, or a temperature reading if the customer happens to have taken one. The associate must accept the first sufficient item offered and must not request a second.

3.4 Tier three, supervisor review. Applies where the claim value exceeds eight hundred rupees, where the claim concerns a foreign object or suspected contamination, where the claim concerns illness, or where the account is a repeat claimant. Tier three cases are routed to a supervisor within fifteen minutes and resolved within four hours. The customer is informed of the review and given an interim acknowledgement, never left in silence.

3.5 Evidence method one, customer photograph. A photograph showing condensation absent from the container lid, congealed ghee or oil on the surface of a rice preparation, or solidified fat on a kurma surface is accepted as sufficient indication of cold service. Associates are not required to make a metallurgical judgement about the photograph. If it plausibly shows cold food, it is cold food.

3.6 Evidence method two, dispatch telemetry. The dispatch system logs the hot box seal timestamp, the hot box internal temperature at seal, the rider pickup timestamp, and the delivery timestamp. Where the elapsed time between seal and delivery exceeds the dish specific hold time in Section 4, the claim is auto-validated regardless of what the customer sends. Associates must check telemetry before asking the customer for anything.

3.7 Evidence method three, batch correlation. Where two or more independent claims of cold food arrive from the same kitchen batch within the same meal period, all subsequent claims from that batch are auto-validated for the remainder of the twenty four hour claim window, and the fulfilment engine proactively issues credits to every customer in that batch who has not complained. This proactive issuance is mandatory, not optional.

3.8 Absence of evidence is never a ground for refusal in tier one. The cost of occasional over-settlement is explicitly accepted by TastyTiffin as lower than the cost of adversarial support interactions.

SECTION 4. FOOD QUALITY AND TEMPERATURE STANDARDS BY DISH

4.1 Every dish has a defined dispatch temperature, a minimum acceptable service temperature at the customer's door, and a maximum hold time from kitchen seal to delivery. A breach of any one of the three constitutes a quality failure attracting the remedies of Section 2.

4.2 Sambar rice. Dispatch temperature seventy eight degrees Celsius. Minimum service temperature fifty five degrees Celsius. Maximum hold time seventy five minutes. Sambar rice is the single highest volume item on the TastyTiffin menu and accounts for roughly twenty six percent of lunch line items. The dish is judged cold if the rice has formed a surface skin, if the sambar has separated into a watery upper layer and a dense lower layer, or if the ghee has solidified into visible white flecks. Sambar rice must be packed in the insulated four hundred millilitre container, never the standard container, irrespective of order size.

4.3 Curd rice. Dispatch temperature between eight and fourteen degrees Celsius. This is a cold-served dish and the cold food clause of Section 2.1 does not apply to it in the ordinary sense. Instead, curd rice fails quality if it is delivered above twenty two degrees Celsius, if it has separated with visible whey pooling above one centimetre in depth, or if it has soured beyond the acceptable pH tolerance. A warm curd rice claim is adjudicated under the same tier structure and attracts the same remedy as a cold food claim. Maximum hold time for curd rice is ninety minutes because the chilled pack degrades faster than the hot box holds.

4.4 Chapati with kurma. Chapati dispatch temperature sixty eight degrees Celsius, kurma dispatch temperature eighty degrees Celsius. Minimum service temperature for chapati is forty eight degrees Celsius and for kurma fifty two degrees Celsius. Maximum hold time seventy minutes. Chapati fails quality independently if it has hardened to the point of cracking when folded, which is the most common chapati complaint and is treated as a cold food claim even where temperature evidence is absent. The kurma must be packed in a separate sealed cup and never poured over the chapati at the kitchen, because saturated chapati is a guaranteed complaint.

4.5 Lemon rice. Dispatch temperature seventy two degrees Celsius. Minimum service temperature fifty degrees Celsius. Maximum hold time eighty minutes. Lemon rice tolerates transit better than sambar rice because it carries less free liquid. It fails quality if the rice has clumped into a single mass, if the peanuts have gone soft, or if the tempering has become bitter, which indicates over-held curry leaves.

4.6 Vegetable biryani. Dispatch temperature eighty two degrees Celsius. Minimum service temperature fifty eight degrees Celsius. Maximum hold time seventy minutes. Vegetable biryani carries the highest average line item value on the menu and correspondingly the highest complaint sensitivity. It fails quality if the rice is gummy, if the vegetables have disintegrated, or if the raita accompanying it has warmed above twenty two degrees Celsius. The raita is treated as part of the biryani line item for refund purposes and is not separately refundable.

4.7 Millet pongal. Dispatch temperature seventy six degrees Celsius. Minimum service temperature fifty four degrees Celsius. Maximum hold time sixty five minutes, the shortest on the menu, because millet stiffens more aggressively than rice on cooling. Millet pongal fails quality if it has set into a sliceable block, if the pepper tempering has sunk entirely to the base, or if it requires water to be edible. The short hold time means millet pongal is not dispatched to zone four addresses during peak lunch, as described in Section 5.

4.8 Rasam rice. Dispatch temperature eighty degrees Celsius. Minimum service temperature fifty six degrees Celsius. Maximum hold time seventy minutes. Rasam rice is the most spill prone item on the menu due to its liquid ratio and accounts for a disproportionate share of meal integrity failures rather than temperature failures. Rasam must be packed in the leak resistant screw cap container and dispatched upright with the biodegradable collar. Any rasam rice delivered with visible leakage into the outer bag is a full line item failure with no evidence tier applied, settled immediately.

4.9 Paneer butter masala with parotta, the Friday special. Paneer butter masala dispatch temperature eighty four degrees Celsius, parotta dispatch temperature seventy degrees Celsius. Minimum service temperature for the gravy is sixty degrees Celsius and for the parotta fifty degrees Celsius. Maximum hold time sixty minutes, enforced strictly. The Friday special carries an enhanced remedy described in Section 4.10 because it is the highest priced item and is ordered by customers who have specifically waited for it.

4.10 Friday special enhanced remedy. Where the Friday special fails any quality standard, the customer receives the full line item refund or credit under Section 2.1, and additionally a guaranteed priority slot for the following Friday at no charge, which reserves capacity even if the Friday special has otherwise sold out. This enhanced remedy may be issued by any associate without approval and does not count toward repeat claimant thresholds.

4.11 Standing packaging rules across all dishes. Every hot line item is sealed within four minutes of plating. Every hot box is loaded with a minimum of two and a maximum of six meal units, because a box loaded beyond six loses seven to eleven degrees Celsius over a thirty minute transit due to repeated opening. Every chilled item travels in the separate chilled sleeve and is never placed inside the hot box, a rule violated often enough during peak dinner that it is listed here as a named audit point.

SECTION 5. DELIVERY WINDOWS, ZONES AND SERVICE COVERAGE

5.1 Lunch service operates from eleven thirty in the morning to two o'clock in the afternoon. Dinner service operates from six thirty in the evening to nine thirty at night. These are the outer service envelopes across all of Chennai. The promised window shown to an individual customer at checkout is always a narrower slot within these envelopes, typically thirty minutes wide.

5.2 Order cut-off times. Lunch orders close at ten forty five in the morning for zone one and zone two, and at ten fifteen for zone three and zone four. Dinner orders close at five forty five in the evening for zone one and zone two, and at five fifteen for zone three and zone four. Orders placed after cut-off are automatically rolled to the next meal period with explicit customer confirmation, never silently.

5.3 Zone one, core. Covers T Nagar, Nungambakkam, Alwarpet, Teynampet, Mylapore, Royapettah, Egmore, Chetpet, Kilpauk and Purasawalkam. Median transit time is eighteen minutes. All menu items are available for both meal periods. Zone one carries the tightest promised windows, typically twenty five minutes wide.

5.4 Zone two, near ring. Covers Adyar, Besant Nagar, Thiruvanmiyur, Velachery, Guindy, Saidapet, Ashok Nagar, K K Nagar, Vadapalani, Anna Nagar, Aminjikarai and Perambur. Median transit time is twenty seven minutes. All menu items available for both meal periods. Promised windows are thirty minutes wide.

5.5 Zone three, outer ring. Covers Perungudi, Thoraipakkam, Sholinganallur, Madipakkam, Pallikaranai, Porur, Ramapuram, Valasaravakkam, Mogappair, Ambattur, Villivakkam and Madhavaram. Median transit time is thirty nine minutes. All menu items available at lunch. At dinner, millet pongal is unavailable in zone three because the sixty five minute hold ceiling cannot be reliably met against evening traffic. Promised windows are forty minutes wide.

5.6 Zone four, periphery. Covers Navalur, Siruseri, Kelambakkam, Poonamallee, Avadi, Tambaram, Chromepet, Pallavaram, Medavakkam and Manapakkam. Median transit time is fifty two minutes. Lunch service only for millet pongal and rasam rice. Full menu at dinner except millet pongal. Promised windows are fifty minutes wide. Zone four orders below a minimum basket value of two hundred and forty rupees are not accepted, because single unit dispatch to periphery is economically and thermally unviable.

5.7 Monsoon protocol. During an amber or red rainfall alert issued for Chennai, promised windows across all zones are extended by twenty minutes at the point of checkout, and the customer sees the extended window before paying. Lateness is measured against the extended window. Where an alert is issued after a customer has already checked out, the original window governs and the lateness ladder applies in full. TastyTiffin absorbs this cost deliberately as a matter of policy.

5.8 Flood and civil disruption suspension. Where a zone is declared undeliverable by the operations control desk, all pending orders for that zone are cancelled with an immediate full refund to source plus a one hundred rupee wallet credit as an inconvenience acknowledgement, issued automatically. No customer contact is required to trigger this.

5.9 Address quality. Where a delivery address lacks a floor number, gate reference, or landmark in a gated community, and the rider is delayed as a result, the lateness ladder still applies for the first such occurrence on that address. From the second occurrence onward on the same saved address, and only after the customer has been notified in writing and the app has prompted for address enrichment, lateness caused by address ambiguity is excluded under Section 6.

SECTION 6. THE LATENESS LADDER AND ITS EXCLUSIONS

6.1 Base commitment restated. Delivery beyond forty five minutes past the promised window earns a twenty percent coupon.

6.2 The full ladder is graduated by severity, because a five minute overrun and a ninety minute overrun are not the same customer experience.

6.3 Rung one, zero to forty five minutes late. No automatic compensation. A proactive in-app apology notification is sent. If the customer contacts support, the associate may issue a fifty rupee wallet credit as goodwill under the Section 14 cap without supervisor approval.

6.4 Rung two, forty six to seventy five minutes late. Twenty percent coupon issued automatically, valid for thirty days, with a maximum discount ceiling of two hundred rupees and no minimum basket condition.

6.5 Rung three, seventy six to one hundred and twenty minutes late. Twenty percent coupon plus fifty percent refund of the order value to source, issued automatically. The customer is not asked to choose and is not asked to justify.

6.6 Rung four, beyond one hundred and twenty minutes late. Full refund of the entire order to source plus a twenty percent coupon plus a supervisor callback within twenty four hours. The food, if it has been delivered, is the customer's to keep or discard as they see fit. TastyTiffin never asks for a rung four meal to be returned.

6.7 Rung five, non-delivery. Where the order is never delivered at all, full refund to source within one banking day, a one hundred and fifty rupee wallet credit, and free delivery on the next three orders. Non-delivery is treated more seriously than extreme lateness because it typically means the customer went without a meal entirely.

6.8 Exclusion one, customer unavailability. Where the rider arrives within the promised window and the customer does not answer after three call attempts spread over eight minutes, the clock stops at the rider arrival timestamp. The lateness ladder does not apply to the waiting period. Section 7 governs what happens to the food.

6.9 Exclusion two, gate refusal. Where a gated community, corporate campus, or hospital security desk refuses rider entry and the customer does not come down, the clock stops at the security refusal timestamp, which the rider must log with a photograph of the gate.

6.10 Exclusion three, address ambiguity after notice, as described in clause 5.9.

6.11 Exclusion four, declared force majeure. Cyclone landfall, city wide curfew, general strike affecting road transport, or a state ordered shutdown. Force majeure must be declared by the operations control desk with a timestamp, and cannot be applied retroactively by an individual associate to justify a specific late delivery.

6.12 Ordinary traffic, ordinary rain, rider shortage, kitchen backlog, order volume surge, a rider taking a wrong turn, a vehicle breakdown, and app or payment gateway downtime are expressly not exclusions. These are TastyTiffin's operational risks and the ladder applies in full. Associates must never cite traffic as a reason to deny a lateness claim. This is a terminable conduct issue, not a coaching issue.

SECTION 7. FAILED HANDOVER, UNREACHABLE CUSTOMERS AND FOOD DISPOSITION

7.1 The rider protocol on arrival is: call attempt one, wait three minutes, call attempt two, wait three minutes, in-app notification and message, wait two minutes, call attempt three. Total waiting obligation is eight minutes from arrival.

7.2 If contact is established within the eight minutes and the customer requests a short additional wait, the rider waits up to five further minutes. Beyond thirteen minutes total, the rider is released and the order enters failed handover status.

7.3 On failed handover, the customer may elect within thirty minutes to have the order redelivered on the next available rider run at a redelivery fee of forty rupees, or may forfeit the order. Where the failure was caused by TastyTiffin, such as a wrong address entered by the support desk during a phone order, the redelivery fee is waived and the redelivered meal is freshly prepared, not the original box.

7.4 Food from a failed handover is never re-dispatched to a different customer under any circumstance. It is logged, removed from circulation, and disposed of or routed to the staff meal programme within the same meal period. Any suggestion in any conversation that a returned or undelivered meal could be given to another customer is a food safety violation and must be escalated immediately.

7.5 Where the customer is unreachable and the delivery address is a residence with a nominated safe drop instruction on file, the rider may complete a contactless drop at the nominated location after the eight minute protocol. In that case the delivery is deemed complete and the cold food clause is measured from the drop timestamp, not from when the customer eventually collects it. Customers electing safe drop are told this explicitly at the point of setting the instruction.

SECTION 8. WALLET CREDIT, COUPONS AND STACKING RULES

8.1 Wallet credit is issued instantly on adjudication and appears in the customer's account within sixty seconds. It never expires. This is a deliberate departure from industry practice and is not to be misstated by associates as time limited.

8.2 Wallet credit is not withdrawable to a bank account and is not transferable between accounts. It survives account dormancy. On account closure requested by the customer, any wallet credit balance above one hundred rupees is refunded to the last used payment instrument on request.

8.3 Wallet credit applies to the food subtotal, packaging charge, and delivery fee. It does not apply to statutory taxes, which are always collected in cash equivalent, nor to the rider tip, which always reaches the rider in full and is never absorbed by credit.

8.4 Coupons carry an expiry of thirty days from issuance unless stated otherwise. Coupons issued as compensation under the lateness ladder carry no minimum basket condition. Marketing coupons may carry a minimum basket condition, which must be disclosed at issuance.

8.5 Stacking rule one. One marketing coupon and one compensation coupon may be used together on a single order. Two marketing coupons may not be combined. Two compensation coupons may not be combined, but the unused one is automatically extended by fifteen days so the customer loses nothing.

8.6 Stacking rule two. Wallet credit stacks with any coupon without restriction. Wallet credit is applied after coupon discount, never before, which is more favourable to the customer.

8.7 Stacking rule three. Subscription pricing is not a coupon and does not conflict with coupon usage. Subscribers may apply compensation coupons to add-on items ordered outside the subscription.

8.8 Refund settlement timelines. Unified payments interface reversals settle within one to three banking days. Credit and debit card reversals settle within five to seven banking days. Net banking reversals settle within three to five banking days. Corporate invoice adjustments appear on the next monthly invoice. Cash on delivery refunds are settled as wallet credit by default, or by unified payments interface transfer to a customer-supplied handle within two banking days if the customer prefers.

8.9 Where a refund has not appeared within the stated timeline, the associate must not tell the customer to contact their bank as a first response. The associate raises a payment trace with the treasury desk, provides the acquirer reference number to the customer, and follows up proactively within twenty four hours.

SECTION 9. SUBSCRIPTIONS, CORPORATE ACCOUNTS AND BULK ORDERS

9.1 Subscription plans available are the five day weekday lunch plan, the five day weekday dinner plan, the six day extended lunch plan, the full week both meals plan, and the fifteen meal flexible pack valid for forty five days.

9.2 Subscription pricing carries a discount of between eight and eighteen percent against à la carte pricing, scaled by plan length. Subscription pricing is locked at the time of purchase and is not affected by menu price revisions during the active cycle.

9.3 Pause. A subscriber may pause deliveries for up to fourteen days per billing cycle with no charge and no loss of meals. Pause must be requested before the cut-off time for the affected meal period. A pause requested after cut-off applies from the following meal period, and the current meal is delivered and consumed from the pack.

9.4 Cancellation. A subscriber may cancel at any time. Unconsumed meals are refunded at the à la carte value minus the discount already enjoyed on consumed meals, and the resulting balance is refunded to source, never forced into wallet credit. Where the calculation produces a negative balance, that balance is written off and never recovered from the customer.

9.5 Subscription quality failures. A cold or defective meal within a subscription is remedied by adding one meal to the pack, or by wallet credit at the à la carte value, at the customer's election. The subscriber is never told that subscription meals carry a lesser remedy, because they do not.

9.6 Corporate accounts. Corporate bulk orders above twenty meal units require confirmation forty eight hours in advance, carry a dedicated dispatch, and are subject to a separate service level of a fifteen minute delivery window rather than the standard window. Lateness beyond fifteen minutes on a corporate dispatch attracts a five percent invoice credit per fifteen minute block, capped at thirty percent of the dispatch value.

9.7 Corporate cancellation. Cancellation more than twenty four hours before the dispatch slot is free. Between twenty four and six hours, a thirty percent charge applies to cover procurement already committed. Under six hours, a seventy percent charge applies. Where TastyTiffin cancels a corporate dispatch for any reason other than declared force majeure, the client receives a full refund plus a credit equal to twenty five percent of the dispatch value.

9.8 Corporate escalation contacts are maintained separately and corporate claims never enter the consumer tier structure of Section 3. A corporate claim is acknowledged within one hour during business hours and resolved within one business day.

SECTION 10. ALLERGENS, DIETARY DECLARATIONS AND SUBSTITUTIONS

10.1 The entire TastyTiffin menu is vegetarian. No dish contains meat, fish, or egg at any stage of preparation, and no shared equipment handles non-vegetarian input, because the kitchen has never processed non-vegetarian input.

10.2 Dairy is present in curd rice, kurma, paneer butter masala, the raita accompanying vegetable biryani, and in the ghee used across rice preparations. A dairy free preparation of sambar rice, lemon rice, and rasam rice is available on request at no additional charge, prepared with cold pressed groundnut oil in place of ghee, subject to being requested at least ninety minutes before the meal period cut-off.

10.3 Groundnut is present in lemon rice as a garnish and in some kurma preparations as a thickening base. Groundnut free variants are available on request with the same ninety minute notice.

10.4 Gluten is present in chapati and parotta, both of which are wheat based. No gluten free flatbread is offered, and associates must not suggest that any flatbread on the menu is gluten free. Millet pongal uses little millet and is naturally gluten free, but is prepared in a kitchen that handles wheat, and this cross contact must be disclosed rather than glossed over.

10.5 Where a customer states a serious allergy, the associate must record it on the account, must repeat the disclosure of cross contact risk, and must never provide reassurance beyond what this section states. The associate does not offer medical opinion, does not estimate risk levels, and does not tell a customer that a trace amount will be fine.

10.6 A meal delivered in violation of a recorded dietary declaration is a tier three claim regardless of value, attracts a full order refund, and triggers a mandatory kitchen incident review with a written outcome shared with the customer within seventy two hours.

10.7 Where a customer reports illness following a meal, the associate must not adjudicate, must not offer or imply a causal admission, and must not request that the customer send photographs of symptoms. The associate immediately escalates to the food safety desk, records the meal period, order identifier, and dishes consumed, offers a full refund without conditions, and provides the food safety desk contact. Refund in this case is issued as an unconditional courtesy and is expressly not an admission of causation.

SECTION 11. HYGIENE, SOURCING AND OPERATIONAL STANDARDS

11.1 Vegetables are procured daily and no vegetable input older than thirty six hours enters preparation. Rice is cooked in batches with a maximum service life of three hours from the point the batch is finished.

11.2 No cooked preparation is carried across meal periods. Lunch surplus is never served at dinner. Dinner surplus is never served the following lunch. This rule has no exception, including on days of unusually low demand.

11.3 Hot boxes are washed and sanitised after every dispatch run. Rider bags are deep cleaned daily and inspected weekly. A rider bag failing inspection is withdrawn from service immediately.

11.4 Oil is filtered after each service and discarded after a maximum of three service periods, tracked by batch card. Reuse beyond three periods is a terminable violation.

11.5 Every kitchen staff member holds a current food handler health clearance. Any staff member reporting fever, gastrointestinal symptoms, or an open wound on the hands is removed from food contact duties for the day with no loss of pay, because a pay penalty would create an incentive to conceal illness.

11.6 Packaging is bagasse based and compostable across all containers except the screw cap rasam container, which is food grade recyclable polypropylene pending a compostable alternative that survives liquid transit.

11.7 Cutlery is supplied only on explicit request at checkout. Absence of cutlery where it was requested is a valid but low severity claim, remedied by a thirty rupee wallet credit issued without question.

SECTION 12. FRAUD, ABUSE AND THE REPEAT CLAIMANT PROTOCOL

12.1 TastyTiffin assumes good faith. The repeat claimant protocol exists to route unusual patterns to human review, not to punish frequent complainants, and associates are prohibited from telling a customer that they complain too often.

12.2 An account crossing three compensable claims in a rolling thirty day period is flagged for review. The review examines whether the claims correlate with a specific kitchen, a specific rider, a specific dish, a specific address, or a specific time slot. In the substantial majority of reviewed cases the pattern is a genuine operational fault, most commonly a single address with a persistent last mile access problem, and the outcome is an operational fix rather than any customer restriction.

12.3 Where review finds no operational correlation and the claim pattern is statistically extreme, the account moves to evidence tier two for all subsequent claims for sixty days. The customer is notified in writing of the change, the reason, and the end date. Silent downgrades are prohibited.

12.4 An account is never blocked, restricted, or closed for claim volume alone by an associate. Only the fraud desk may restrict an account, only with a written case file, and only after the customer has had an opportunity to respond.

12.5 Promotion abuse, meaning systematic creation of multiple accounts to reuse first order offers, is handled by the fraud desk under a separate policy and is not within the scope of associate discretion.

12.6 Abusive conduct toward riders or associates, including threats, slurs, or sexual harassment, is handled under the safety policy, not this handbook. A claim raised by a customer who has behaved abusively is still adjudicated on its merits under this handbook. The two matters are kept strictly separate.

SECTION 13. ESCALATION MATRIX AND RESPONSE TIMES

13.1 Level zero, self service. In-app claim flow, resolves tier one claims automatically without human contact, target resolution under thirty seconds.

13.2 Level one, associate. Handles tier one and tier two claims, may issue up to eight hundred rupees in refund or credit per claim and up to fifty rupees in goodwill without approval.

13.3 Level two, senior associate. Handles tier two and routine tier three, may issue up to two thousand rupees per claim and up to two hundred rupees in goodwill. Target first response within fifteen minutes.

13.4 Level three, supervisor. Handles foreign object claims, dietary declaration violations, corporate claims, and any claim above two thousand rupees. Target first response within one hour, resolution within four hours.

13.5 Level four, food safety desk. Handles illness reports, contamination reports, and regulatory contact. Target first response within thirty minutes at any hour, seven days a week.

13.6 Level five, customer experience council. Handles regulatory complaints, consumer forum notices, media enquiries, and any case where the customer states an intention to pursue legal remedy. Associates route these immediately and do not attempt to negotiate.

13.7 No claim may sit unacknowledged for more than four hours in any channel. An acknowledgement that states the next concrete step and its timing is required even where the resolution is pending.

SECTION 14. UNSPECIFIED SCENARIOS AND GOODWILL

14.1 Where a scenario is not covered by this handbook, the associate does the following in order. First, acknowledge the customer's account of what happened without disputing it. Second, apply the nearest analogous clause if one plausibly exists, and say which one. Third, if no clause applies, issue goodwill within the associate's Section 13 limit and route the case to a supervisor with a short note describing the gap.

14.2 Gap notes are reviewed weekly by the customer experience council. Any scenario appearing three times in a quarter becomes a numbered clause in the next revision. This is the sole mechanism by which this handbook grows, and it means the absence of a clause is a defect in the handbook rather than a defect in the customer's claim.

14.3 Goodwill is capped per associate level and logged against the associate identifier for pattern analysis only. Goodwill issuance is never used as a negative performance metric, because that would create pressure to refuse legitimate claims.

SECTION 15. ASSOCIATE SCRIPTS AND LANGUAGE STANDARDS

15.1 Opening for a quality claim. Acknowledge, apologise once, state the remedy, ask for the settlement preference. Do not apologise repeatedly, which reads as evasion, and do not ask what the customer would like as a resolution before stating what they are entitled to.

15.2 Prohibited phrasings. Do not say that the food was fine when it left the kitchen. Do not say that traffic was heavy. Do not say that this is company policy without stating what the policy actually provides. Do not say that the customer should have collected the order faster. Do not ask whether the customer has already eaten the food, which is irrelevant to entitlement under every clause of this handbook.

15.3 Required disclosures. When issuing wallet credit, state that it never expires. When issuing a coupon, state the expiry and the discount ceiling. When issuing a refund to source, state the expected settlement window for that specific payment instrument.

15.4 Language. Service is provided in Tamil, English, Telugu, Hindi and Malayalam across the support desk. The customer selects the language and the associate follows it without comment. Policy meaning must not vary across languages, and translated scripts are maintained centrally rather than improvised.

15.5 Closing. Every claim interaction closes with a plain statement of what has been done, what will happen next, and by when. No claim closes with an open ended assurance that someone will look into it.

SECTION 16. WORKED EXAMPLES

16.1 Example one. A customer in Adyar orders sambar rice and curd rice for lunch with a promised window of twelve thirty to one o'clock. Delivery timestamp is one fifty two. The sambar rice is cold. Adjudication: lateness is fifty two minutes past the closing edge, placing it at rung two, so a twenty percent coupon issues automatically. The cold sambar rice is separately refunded in full as a tier one claim on the customer's word. The curd rice, being cold-served, is unaffected and is not refunded. Total outcome: one line item refunded, one coupon issued.

16.2 Example two. A customer in Sholinganallur orders millet pongal for dinner. The order is rejected at checkout because millet pongal is unavailable in zone three at dinner under clause 5.5. The customer contacts support arguing that they received it last month. Adjudication: the restriction is current policy; the associate explains the hold time reason plainly, offers rasam rice or sambar rice as the nearest alternative, and does not issue compensation because no service failure has occurred. If the customer had in fact been served millet pongal in zone three at dinner recently, the associate logs a gap note under clause 14.1 because that indicates a dispatch control failure.

16.3 Example three. A corporate client in Guindy orders forty units for a twelve thirty dispatch. Delivery completes at one o'clock. Adjudication: corporate service level is a fifteen minute window, so this is two fifteen minute blocks late, producing a ten percent invoice credit. The consumer lateness ladder does not apply and the twenty percent coupon is not issued, because corporate remedies are exclusive under Section 9.6.

16.4 Example four. A customer reports a foreign object in vegetable biryani. Adjudication: tier three regardless of order value under clause 3.4. Immediate full order refund, no evidence demanded before refunding, escalation to level four under clause 13.5, kitchen batch quarantine, and a written outcome to the customer. The associate does not ask the customer to keep the object, though if the customer offers it, collection is arranged.

16.5 Example five. A customer's Friday special arrives at sixty two minutes from kitchen seal, two minutes over the hold ceiling, but the customer says it was hot and is happy. Adjudication: no claim exists, nothing is issued, and the customer is not contacted about it. The telemetry breach is logged for kitchen review only. Policy does not manufacture claims the customer has not made.

16.6 Example six. A customer in Tambaram places a two hundred rupee order at eleven o'clock for lunch. Adjudication: the order is below the zone four minimum basket of two hundred and forty rupees under clause 5.6 and also after the zone four lunch cut-off of ten fifteen under clause 5.2. Both conditions block the order. The customer is offered dinner service instead with the correct cut-off stated.

16.7 Example seven. A subscriber pauses on a Tuesday at eleven o'clock for that day's lunch. Adjudication: lunch cut-off for their zone two address was ten forty five, so the pause applies from dinner onward. Tuesday lunch is delivered and consumed from the pack. The associate states this clearly rather than allowing the subscriber to assume the meal was saved.

16.8 Example eight. Rasam rice arrives leaked into the outer bag, soaking a chapati ordered alongside it. Adjudication: the rasam rice is a full line item failure settled immediately under clause 4.8 with no evidence tier. The chapati is a consequential meal integrity failure under clause 1.10 and is also refunded in full. Two line items refunded, no lateness element, no coupon.

SECTION 17. FREQUENTLY ASKED QUESTIONS

17.1 Does a refund require returning the food. No. TastyTiffin never asks for food to be returned for a refund.

17.2 Does wallet credit expire. No, wallet credit never expires. Coupons expire in thirty days.

17.3 Can a late order and a cold order be compensated together. Yes, always. The two remedies are independent under clause 2.3.

17.4 What if the twenty four hour claim window has just passed. The claim moves to supervisor discretion under Section 14 rather than being refused outright. Where the customer's account of the delay is plausible, such as a meal ordered for someone else who reported the problem late, the claim is settled.

17.5 Is the delivery fee refunded. Yes, wherever the order or a majority of its value is refunded, the delivery fee is refunded with it. The rider tip is never clawed back.

17.6 Can a customer pick up from the kitchen. No. TastyTiffin operates as a delivery only kitchen with no counter service, and customers must not be directed to a kitchen address.

17.7 What happens on a public holiday. Service continues on all public holidays except Pongal day and Deepavali day, when the kitchen closes and subscriptions are automatically extended by one day at no charge.

17.8 Does the Friday special ever change. The Friday special is paneer butter masala with parotta as standing policy. Any substitution requires a menu revision to this handbook and advance notice to subscribers, and is never a same day decision.
"""

big_policy = "You are Mitra, TastyTiffin's support assistant. Follow these policies.\n\n"
big_policy += policy_chunk    # Comfortably exceed the 4,096-token minimum

count = client.messages.count_tokens(
    model=MODEL,
    system=big_policy,
    messages=[{"role": "user", "content": "hi"}],
)
print("Prompt size:", count.input_tokens, "tokens (need > 4096 to cache on Haiku 4.5)")

In [ ]:
# Lab 5b: call twice — first call WRITES the cache, second call READS it

def ask_mitra(question):
    reply = client.messages.create(
        model=MODEL,
        max_tokens=100,
        system=[{
            "type": "text",
            "text": big_policy,
            "cache_control": {"type": "ephemeral"},   # breakpoint: cache up to HERE
        }],
        messages=[{"role": "user", "content": question}],
    )
    u = reply.usage
    print("  cache WRITE tokens :", u.cache_creation_input_tokens)
    print("  cache READ  tokens :", u.cache_read_input_tokens)
    print("  normal input tokens:", u.input_tokens)
    print("  Mitra:", reply.content[0].text, "\n")

print("--- Call 1 (expect a big WRITE, zero READ) ---")
ask_mitra("My order #4521 arrived cold. What are my options?")

print("--- Call 2 (expect zero WRITE, big READ = 90% off) ---")
ask_mitra("Is veg biryani on the menu?")

**Expected result:**

- **Call 1:** `cache_creation_input_tokens` ≈ several thousand (written at 1.25×), `cache_read_input_tokens` = 0.
- **Call 2:** `cache_creation_input_tokens` = 0, `cache_read_input_tokens` ≈ the same several thousand — now billed at **USD 0.10/MTok instead of USD 1.00**. The tiny `input_tokens` is just the new question.

**Try this 🔧:** (1) Wait 6+ minutes and call again — the cache expired, so you'll see a fresh WRITE. (2) Change one word inside `big_policy` and call — cache miss, fresh WRITE. Exact match matters.

> **Also good to know:** the docs' newer *automatic caching* option puts `cache_control={"type": "ephemeral"}` at the **top level** of the request, and the API auto-places the breakpoint on the last cacheable block — ideal for growing multi-turn conversations. We used an **explicit** breakpoint here because our stable part (the policy) is followed by a question that changes every call, and the breakpoint must sit on content that *doesn't* change. Knowing when to use which is an architect-level detail.

### ✅ Checkpoint

> **Q:** Priya's nightly batch job runs once per day with the same giant system prompt. Will the 5-minute cache help across runs on different days?

<details><summary>Show answer</summary>

**No.** The cache lives 5 minutes (or 1 hour with the paid `ttl: "1h"` option). Runs a day apart always re-write the cache. Caching helps *bursts* of similar requests, not widely spaced ones.
</details>

### 🎤 Interview angle

**Q (FDE): "A client says: 'Caching sounds like it stores our customer data on your servers — compliance won't allow it.'"**
*Model answer:* "The cache holds the model's processed state for a prefix, in memory, for minutes — with cryptographic hashes for matching, isolated per organization (and per workspace), and it's even eligible under Anthropic's zero-data-retention terms. I'd frame it as a short-lived performance layer, not data storage, and show the compliance team the docs' data-retention section."


# Section 7 — How Real Products Do It (and the claude.ai Bridge)

Everything you built today runs inside the biggest AI products, right now:

| What you built today | Where you've already seen it |
|---|---|
| Conversation list replayed each turn | Every chat on **claude.ai** — the app resends your thread to the API each message |
| Facts pinned so they survive trimming | **claude.ai Projects** (project knowledge sent with every chat) and memory features |
| Sliding window / trimming | Long claude.ai chats eventually hit a length limit — same context window you measured |
| Prompt caching | claude.ai, Claude Code, and most serious API products cache system prompts and tools under the hood |

**Beyond today (know these exist — one line each):**

- **Summarization memory:** ask the model to summarize dropped turns and keep the summary in context — recall without the token bill.
- **Anthropic's memory tool + context editing:** newer API features where Claude itself reads/writes memory files across conversations and old tool results are auto-cleared. Built from the same primitives you just learned.
- **RAG (a later session):** store knowledge in a database and retrieve only the relevant bits per request — memory that scales beyond any context window.

> If you understand today's list + window + cache, none of those will ever feel like magic — they're the same three ideas, industrialized.


# Architecture — Mitra in Production

```
 Customer (app / WhatsApp)
        |
        v
 +--------------------+     load/save history      +------------------+
 |  Priya's backend   | <------------------------> |  Database        |
 |  (the "state" home)|      per customer          |  (chat history,  |
 |                    |                            |   facts/profile) |
 |  1. load history   |                            +------------------+
 |  2. sliding window |
 |  3. build request: |
 |     [cached: system+policies] + [history] + [new msg]
 |  4. call Claude    |
 +---------|----------+
           v
   Claude API (STATELESS)
   - prompt cache (5 min): policies read at 0.1x price
   - 200K context window (Haiku 4.5)
```

**Component responsibilities:** the backend owns *state* (load, trim, save); the database owns *durability* (history survives restarts); the Claude API owns *intelligence* (and holds zero conversation state).

**Failure points an architect names in review:**
- Database down → bot still answers but with amnesia. Decide: fail closed or degrade gracefully?
- Window trims a critical fact → wrong answers. Mitigate: pin facts to the system prompt / profile store.
- Someone "just edits" the system prompt at noon → every cache misses at once → cost and latency spike. Mitigate: version prompts, deploy in low-traffic windows.
- Trimming bug breaks user-first/alternating order → hard API errors. Mitigate: pair-safe trimming + a validation check before send.

**Cost levers, in the order an architect pulls them:** smaller model (Haiku) → prompt caching → tighter window → summarization → shorter system prompt.


# 🧪 Main Claude API Lab — Mitra v3: Memory + Window + Cache Together

**Objective:** combine all three techniques in one production-shaped bot, and print the cache receipt every turn.


In [ ]:
# Mitra v3 — the full session in ~35 lines

class MitraBotV3:
    def __init__(self, system_text, max_messages=8):
        self.system_blocks = [{
            "type": "text",
            "text": system_text,
            "cache_control": {"type": "ephemeral"},   # cache the stable prefix
        }]
        self.history = []
        self.max_messages = max_messages

    def chat(self, user_text):
        self.history = sliding_window(self.history, self.max_messages)  # from Lab 4
        self.history.append({"role": "user", "content": user_text})
        reply = client.messages.create(
            model=MODEL,
            max_tokens=200,
            system=self.system_blocks,
            messages=self.history,
        )
        answer = reply.content[0].text
        self.history.append({"role": "assistant", "content": answer})
        u = reply.usage
        print(f"  [receipt] write={u.cache_creation_input_tokens} "
              f"read={u.cache_read_input_tokens} normal={u.input_tokens}")
        return answer

mitra3 = MitraBotV3(big_policy)   # the >4096-token policy pack from Lab 5

for q in [
    "Hi, I'm Karthik. Order #4521 arrived cold.",
    "What refund options do I have?",
    "I'll take the wallet credit. Also, is millet pongal available at dinner?",
    "Great. Summarize everything we agreed today.",
]:
    print("Karthik:", q)
    print("Mitra :", mitra3.chat(q), "\n")

**Expected result:** turn 1 shows a cache **write**; turns 2–4 show cache **reads** (90% off the policy pack) while memory works across all turns — the final summary should mention the cold order, #4521, and the wallet credit.

**Try this 🔧:** drop `max_messages` to 2 and rerun. Watch the receipt stay cheap but the final summary lose early details. You are now *tuning* the memory/cost tradeoff — that's the actual day job.

### 💰 A one-line cost note

This whole lab — a dozen Haiku calls with a ~5K-token cached prompt — costs about a cent. Caching is why.


# 🚀 Mini Project — TastyTiffin Support Copilot

**Business use case:** Priya wants one bot that handles a full support shift: greets customers, remembers each customer separately, survives 30+ turn conversations, and keeps API spend flat as traffic grows.

**Architecture (build exactly this):**

```
customers (many) --> SupportDesk
                      - bots: dict  {customer_id -> MitraBotV3}
                      - one shared cached policy prompt (write once, read cheap)
                      - per-customer history + pair-safe window
```

**Build steps:**
1. Wrap `MitraBotV3` in a `SupportDesk` class holding a `dict` of bots keyed by customer ID — memory isolation per customer (Karthik's refund must never leak into Divya's chat).
2. Simulate two interleaved customers (alternate their messages) and prove isolation: ask each "what did I complain about?"
3. Add fact-pinning: after each turn, if the message contains an order number, append it to that customer's system block — so it survives the window. (Careful: what does editing the system text do to the cache? You know the answer now.)
4. Print a per-conversation cost report from the usage receipts: total write, read, and normal tokens → estimated ₹/$ cost.

**Definition of done:** two customers chat 6+ turns interleaved with correct isolated memory; cache reads appear from each customer's second turn; the cost report shows cached turns ≈ 90% cheaper on the policy portion; a 3-sentence written answer to: "what breaks first at 100× traffic?"


# 📋 Session Summary

Claude's API is **stateless**: every request stands alone, so "memory" must be built by your application. The universal pattern is the **conversation list** — resend every user and assistant message each turn. That list grows, colliding with the **context window** (200K tokens on Haiku 4.5) and with **per-token billing on every call**. The **sliding window** caps growth by dropping the oldest user+assistant *pairs* (never break the user-first, alternating rule) at the price of forgetting old facts — so durable facts get pinned in the system prompt or a profile store. **Prompt caching** attacks the other half of the bill: mark the stable prefix with `cache_control` and re-reads cost 0.1× (90% off) with much lower latency, for 5 minutes per refresh (1 hour paid), minimum 4,096 tokens on Haiku 4.5. Memory is your job; caching is your discount; the window is your budget.


# ✅ What You Learned Today

You can now:

- [ ] Explain stateless vs stateful — and *whose job* conversation memory is
- [ ] Prove statelessness with two API calls
- [ ] Implement memory with a `messages` list and the role/content format
- [ ] Build a chatbot class (Mitra) with a system prompt and growing history
- [ ] Count tokens exactly with `client.messages.count_tokens(...)`
- [ ] Explain why cost grows every turn with list-based memory
- [ ] Implement a pair-safe sliding window and name its failure mode
- [ ] Enable prompt caching with `cache_control` and read the usage receipt
- [ ] State the caching numbers: 1.25× write, 0.1× read, 5-min TTL, 4,096-token minimum on Haiku 4.5
- [ ] Sketch the production architecture: backend owns state, DB owns durability, API owns intelligence


# 🗂️ AI Architect Cheat Sheet

**Definitions**

| Term | One-liner |
|---|---|
| Stateless | Server keeps nothing between requests; each request self-contained |
| Stateful | Information carried across requests (your app's job) |
| Context window | Max tokens per request — 200K (Haiku 4.5); 1M on Sonnet 4.6/5 & Opus 4.8 |
| Sliding window | Keep last N messages, drop oldest pairs |
| Prompt caching | Server reuses processed stable prefix; 5-min default TTL |

**Key numbers (Haiku 4.5, verified July 2026)**

| Item | Value |
|---|---|
| Input / output price | 1 USD / 5 USD per MTok |
| Cache write / read | 1.25 USD (1.25×) / 0.10  USD (0.1×) per MTok |
| Cache TTL | 5 min default; 1 hour at 2× write price |
| Min cacheable prompt | 4,096 tokens (Haiku 4.5) — silently skipped below |
| Max cache breakpoints | 4 |
| Token rule of thumb | 1 token ≈ 3.5 English chars ≈ 3/4 word |

**Decision table**

| Symptom | Reach for |
|---|---|
| Bot forgets between turns | Conversation list (send history) |
| Hitting context limit / cost creeping per turn | Sliding window (+ summarize or pin facts) |
| Same big prefix sent repeatedly | Prompt caching (breakpoint on stable content) |
| Facts must survive forever | Pin to system prompt / DB profile (later: memory tool, RAG) |

**API quick-reference**

```python
client.messages.create(model=..., max_tokens=..., system=..., messages=[...])
client.messages.count_tokens(model=..., messages=[...])          # free, exact
system=[{"type":"text","text":BIG,"cache_control":{"type":"ephemeral"}}]
reply.usage.cache_creation_input_tokens / cache_read_input_tokens / input_tokens
```


# ⏱️ 5-Minute Revision Guide

1. **Claude forgets by design.** The API is stateless — for privacy, scale, reproducibility. Memory is the app's job.
2. **The fix is a list.** Store every `{"role","content"}` message; resend all of it each call. User first, roles alternate.
3. **The list bites back.** You re-pay input tokens for the whole history every turn; the context window (200K on Haiku 4.5) is the ceiling.
4. **Sliding window = trim in pairs.** Caps cost; forgets old facts — pin important ones to the system prompt.
5. **Caching = 90% off the stable prefix.** `cache_control: ephemeral`; write 1.25×, read 0.1×; 5-min TTL; exact-match only; ≥4,096 tokens on Haiku 4.5; verify via `usage`.
6. **The one sentence:** *memory is your job, caching is your discount, the window is your budget.*


# 🎤 Interview Preparation Notes

**Q1. Is the Claude API stateful or stateless, and why does it matter?**
Stateless — each request is independent and must contain the full conversation. It matters because memory, cost control, and context management all become application-layer responsibilities.

**Q2. How do you implement multi-turn memory?**
Maintain an ordered list of user/assistant messages, append both sides every turn, resend the entire list. Persist it in a database keyed by user/session for durability.

**Q3. Conversation exceeds the context window — what are your options and tradeoffs?**
Sliding window (cheap, forgets old facts), summarization (recall at small token cost, adds a model call), fact-pinning to system prompt/profile (durable, needs extraction logic), and eventually RAG. Production systems combine them.

**Q4. Explain prompt caching pricing and when it pays off.**
Write costs 1.25× base input, reads cost 0.1×. It pays off from the very first re-read within the TTL (5 min default, 1 h at 2×). Best for stable prefixes reused in bursts: system prompts, tools, policies, long documents, growing chat history.

**Q5. Why can a cache silently not work?**
Below the model's minimum (4,096 tokens on Haiku 4.5), prefix not byte-identical, breakpoint placed on content that changes each request, or TTL expired. Diagnose with the `usage` fields: both cache numbers 0 → nothing cached.

**Q6 (architecture). Design memory for a support bot at 1M conversations/month.**
Backend loads history from a DB per conversation, applies pair-safe windowing + summary, pins durable facts to a profile, marks the shared system/policy prefix with a cache breakpoint, monitors usage receipts for cache hit-rate, and versions prompt changes to avoid cache stampedes.

**Q7 (FDE). Client: "the bot forgot what the user said 40 turns ago — your product is broken."**
Reframe: the model reads only what we send; we tuned the window for cost. Options with price tags: widen the window, add summarization, pin key facts. Demo the fix live with the usage receipt showing the cost impact of each.

**Q8 (FDE). Client worries caching stores their data.**
Cache = short-lived in-memory processed state + hashes, isolated per org/workspace, minutes-long TTL, ZDR-eligible. Show the docs' data-retention section; position as performance layer, not storage.


# 📝 Assignment

**Beginner —** Rebuild `MitraBot` from scratch without looking, for a different business (a gym's front-desk bot). Prove memory with a 3-turn chat.

**Intermediate —** Add `count_tokens` to `MitraBotV2` so it prints per-turn input size, and trigger the sliding window by *token count* (e.g., > 1,500 tokens) instead of message count. Keep it pair-safe.

**Advanced —** Implement **summarization memory**: when the window trims messages, send the dropped turns to Haiku with "summarize in 2 sentences", and keep the running summary as the first user message. Show the bot answering a question about a trimmed turn.

**Project —** Complete the TastyTiffin Support Copilot (mini project spec above), including the per-customer cost report and the "what breaks at 100× traffic" write-up.


# 🧪 Assessment

### Part A — Multiple choice (10)

**1.** "The Claude API is stateless" means:
(a) It can't follow instructions (b) Each request is independent; no conversation is stored between calls (c) It has no system prompt (d) It forgets mid-response

**2.** To make a chatbot remember, your app must:
(a) Enable `memory=True` (b) Use a bigger model (c) Resend the full message history each call (d) Call the same server each time

**3.** A valid `messages` list must:
(a) Start with an assistant message (b) Start with a user message, roles alternating (c) Contain only user messages (d) Be under 10 messages

**4.** Claude Haiku 4.5's context window is:
(a) 4,096 tokens (b) 32K (c) 200K (d) Unlimited

**5.** With list-based memory, the input cost of turn 20:
(a) Equals turn 1's cost (b) Includes tokens from all previous turns (c) Is free after caching (d) Only counts output tokens

**6.** A sliding window should trim:
(a) Newest messages (b) Random messages (c) Oldest user+assistant pairs (d) Only assistant messages

**7.** Prompt caching's read price is:
(a) Free (b) 0.1× base input (c) 1.25× base input (d) 2× base input

**8.** The default cache lifetime is:
(a) 5 minutes (b) 1 hour always (c) 24 hours (d) Forever

**9.** On Haiku 4.5, a 500-token system prompt marked with `cache_control`:
(a) Caches normally (b) Throws an error (c) Is silently not cached — below the 4,096-token minimum (d) Caches at 2× price

**10.** `cache_read_input_tokens: 5200, cache_creation_input_tokens: 0` means:
(a) Cache miss (b) Cache expired (c) 5,200 tokens were reused from cache at 90% off (d) 5,200 tokens were written

### Part B — Short answer (5)

**11.** Why does claude.ai "remember" your conversation when the API is stateless?
**12.** Name the two separate problems caused by an ever-growing history list.
**13.** Why must a sliding window trim in pairs?
**14.** Why should the cache breakpoint never sit on the user's newest message?
**15.** Caching vs memory: one sentence on the difference.

### Part C — Scenarios (3)

**16.** Priya edits one sentence of the cached policy prompt at 12:05 pm during lunch rush. Predict the next 5 minutes of cost and latency, and propose a safer deployment approach.

**17.** A customer chats for 60 turns. Around turn 40, Mitra starts contradicting things agreed in turns 1–10, and by turn 55 the API rejects a request outright. Diagnose both symptoms and propose a combined fix.

**18.** Finance says the bot's API bill doubled after "someone improved the system prompt". The prompt grew from 3,900 to 6,000 tokens — on Sonnet 4.6 calls it got *cheaper per call*, but the Haiku 4.5 fleet got more expensive. Wait — explain why growing a prompt could ever *reduce* cost, and what the Haiku fleet's problem might be. *(Hint: minimum cacheable sizes: Sonnet 4.6 = 1,024; Haiku 4.5 = 4,096.)*


# 🔑 Answer Key

**Part A:** 1-b · 2-c · 3-b · 4-c · 5-b · 6-c · 7-b · 8-a · 9-c · 10-c

**Part B:**
**11.** The claude.ai *application* stores your thread and resends it to the API with every message — app-layer statefulness over a stateless API.
**12.** (1) Rising cost: full history is re-billed as input every call. (2) A hard ceiling: the context window eventually rejects or truncates the request.
**13.** The API expects user-first, alternating roles; trimming an odd number can leave the list starting with an assistant message → errors.
**14.** The newest message changes every request, so the prefix hash never matches → you pay cache *writes* every call and never get a read. Breakpoint belongs at the end of the *stable* content.
**15.** Memory decides *what the model sees* (your app's job); caching makes *re-sending it cheaper and faster* (Anthropic's discount) — caching stores nothing for you long-term.

**Part C:**
**16.** Every request's cached prefix now mismatches → all traffic pays fresh 1.25× cache writes with full processing latency until the new prefix is warm; during rush this is a visible cost + latency spike. Safer: version prompts, deploy during a low-traffic window, or pre-warm the cache before switching traffic.
**17.** Turn-40 contradictions: the sliding window (or truncation) dropped early turns, so agreements from turns 1–10 vanished. Turn-55 rejection: history finally exceeded the context window (or broke message-order rules after a bad trim). Combined fix: pair-safe window + running summary of dropped turns + pin durable facts (order #, promises) to the system prompt/profile.
**18.** At 3,900 tokens the prompt was *below* Haiku 4.5's 4,096-token minimum — it was never cached, every call paid full price. Growing it past 4,096 would actually *enable* caching (as it already did on Sonnet 4.6, whose minimum is 1,024). If the Haiku fleet still got pricier, likely causes: the breakpoint sits on changing content, calls are >5 min apart (TTL expiry), or the prompt isn't byte-identical across servers. The lesson: a *bigger* prompt that caches can be cheaper than a *smaller* one that doesn't — check the usage receipts, not the prompt length.
